In [ ]:
%pip install --upgrade --force-reinstall \
    "pydantic>=2.12,<2.13" \
    langchain \
    langchain-openai \
    langchain-chroma \
    langchain-docling \
    langchain-community \
    langchain-text-splitters \
    chromadb \
    docling \
    beautifulsoup4 \
    "numpy<2" \
    "pandas>=2.2,<3"

In [ ]:
import sys
!{sys.executable} -m pip install -U langchain-community

In [ ]:
import importlib.metadata as md

for package in [
    "docling",
    "langchain-docling",
    "pydantic",
    "langchain",
    "numpy",
]:
    print(package, md.version(package))

from langchain_docling import DoclingLoader

print("Docling import successful")

In [ ]:
import sys
import importlib.metadata as md

print("Python:", sys.executable)

for package in ["docling", "langchain-docling", "pydantic", "pydantic-core"]:
    try:
        print(f"{package}: {md.version(package)}")
    except md.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

In [ ]:
import sys
import numpy as np

print(sys.executable)
print(np.__version__)
print(np.__file__)

In [ ]:
import sys

!{sys.executable} -m pip install \
    --upgrade \
    --force-reinstall \
    --no-cache-dir \
    "pydantic==2.13.5"

In [ ]:
import sys
import pydantic
import importlib.metadata as md

print("Python:", sys.executable)
print("Pydantic:", pydantic.__version__)
print("Pydantic location:", pydantic.__file__)
print("Docling:", md.version("docling"))
print("LangChain Docling:", md.version("langchain-docling"))

from langchain_docling import DoclingLoader

print("Docling import successful")

In [ ]:
import sys
!{sys.executable} -m pip check

### Main imports

In [ ]:
import os
import warnings
import logging

import bs4

from langchain.agents import AgentState, create_agent
from langchain.messages import MessageLikeRepresentation
from langchain.tools import tool

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_docling import DoclingLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

### Environment setup

For this step you will need to: 
- get a Langchain API Key (https://docs.langchain.com/oss/python/deepagents/rag)
- be added DHInfra project by Florian and get DHInfa API kez

In [ ]:
from openai import OpenAI

In [ ]:
os.environ["LANGCHAIN_API_KEY"] = "" # insert your own Langchain key
os.environ["LANGCHAIN_TRACING_V2"] = "false"  # <-- FIX 1: Disabled to prevent 403 error
os.environ["LANGCHAIN_PROJECT"] = "DHInfra-Tracing-Demo"
os.environ["LANGSMITH_DISABLE_RUN_COMPRESSION"] = "true"
os.environ["USER_AGENT"] = "my_agent"
os.environ["DHINFRA_API_KEY"] = "" # insert DHInfra key

In [ ]:
# <-- FIX 2: Custom class to prevent the 422 "null content" error
class SanitizedChatOpenAI(ChatOpenAI):
    def _get_request_payload(self, input_, *args, **kwargs):
        payload = super()._get_request_payload(input_, *args, **kwargs)
        if "messages" in payload:
            for msg in payload["messages"]:
                if msg.get("content") is None:
                    msg["content"] = ""
        return payload

# Initialize chat model using the sanitized class
model = SanitizedChatOpenAI(
    model="qwen3.5-397b",
    openai_api_key=os.environ["DHINFRA_API_KEY"],
    openai_api_base="https://api.dhinfra.uni-graz.at/v1",
    model_kwargs={"parallel_tool_calls": False}
)



# Initialize embedding model
embeddings = OpenAIEmbeddings(
    model="qwen3-embedding-8b",
    openai_api_key=os.environ["DHINFRA_API_KEY"],
    openai_api_base="https://api.dhinfra.uni-graz.at/v1"
)


vector_store = Chroma(
    collection_name="data",
    embedding_function=embeddings,
    persist_directory="./chroma_spectators", # <-- Rename the persistent directory where the DB will live if necessary
)


print("Chat model (Qwen), embedding model (Qwen3-Embedding-8B), and Chroma vector store setup done")

In [ ]:
# Skip re-parsing/re-embedding the whole corpus if persist directory exist
# already contains data (e.g. from a previous run of this notebook).
existing_count = vector_store._collection.count()
NEEDS_INGESTION = existing_count == 0

if existing_count > 0:
    print(f"Found existing Chroma collection with {existing_count} chunks — skipping ingestion.")
else:
    print("No existing data found — will build the collection from scratch.")


## Adapting the code for our own data

In [ ]:
import os
import glob
import warnings

import xml.etree.ElementTree as ET

warnings.filterwarnings("ignore")
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores.utils import filter_complex_metadata

TEI_NS = {"tei": "http://www.tei-c.org/ns/1.0"}

# Directory containing the TEI/XML source files (searched recursively, so
# language- or periodical-specific subfolders are fine).
CORPUS_DIR = "data"  # <-- adjust to wherever the .xml files live

# Label used for running text that falls outside any explicit
# <milestone unit="form"> span (headers, signatures, unmarked prose).
UNMARKED_FORM_LABEL = "unmarked"


def get_text(elem):
    """Return the stripped, concatenated text content of an XML element, or None."""
    if elem is None:
        return None
    text = "".join(elem.itertext()).strip()
    return text or None


def parse_tei_header(root):
    """
    Extract the per-document metadata fields that apply to every chunk
    from this file: Author, date, periodical_names, language.
    """
    author = get_text(
        root.find(".//tei:fileDesc/tei:titleStmt/tei:author", TEI_NS)
    ) or "unknown"

    structured_bibl = root.find(
        ".//tei:fileDesc/tei:sourceDesc/tei:bibl[@type='structured']", TEI_NS
    )
    date = (
        get_text(structured_bibl.find("tei:date", TEI_NS))
        if structured_bibl is not None
        else None
    )
    periodical_name = ( 
        get_text(structured_bibl.find("tei:title[@level='j']", TEI_NS))
        if structured_bibl is not None
        else None
    )

    language_elem = root.find(".//tei:profileDesc/tei:langUsage/tei:language", TEI_NS)
    language = (
        get_text(language_elem)
        or (language_elem.get("ident") if language_elem is not None else None)
        or "unknown"
    )

    return {
        "Author": author,
        "date": date or "unknown",
        "periodical_names": periodical_name or "unknown",
        "language": language,
    }


def _form_vocabulary(root):
    """
    lookup built from the document's own narrative-forms vocabulary: encodingDesc/editorialDecl/interpretation//interpGrp[@type='forms'],
    e.g. {"FP": "Heteroportrait", "EX": "Example", "ZM": "Citation/Motto", ...}.
    """
    code_to_label = {}
    for interp_grp in root.findall(
        ".//tei:encodingDesc/tei:editorialDecl/tei:interpretation//tei:interpGrp[@type='forms']",
        TEI_NS,
    ):
        for interp in interp_grp.findall("tei:interp", TEI_NS):
            code = interp.get("{http://www.w3.org/XML/1998/namespace}id")
            label = get_text(interp)
            if code and label:
                code_to_label[code] = label
    return code_to_label


def extract_form_segments(root):
    """
    Walk the document body in reading order and split it into segments,
    each associated with a single narrative form.

    Narrative-form spans as a milestone/anchor pair rather
    than as nesting elements: <milestone unit="form" ana="#CODE"
    spanTo="#anchorID"/> marks where a form's text begins, and
    <anchor xml:id="anchorID"/> marks where it ends..

    Returns a list of (form_label, text) tuples in document order, with
    consecutive text belonging to the same form merged into one segment.
    Text outside any form span is labelled UNMARKED_FORM_LABEL rather
    than dropped.
    """
    code_to_label = _form_vocabulary(root)
    body = root.find(".//tei:text/tei:body", TEI_NS)
    if body is None:
        return []

    segments = []          # list of [form_label, [text_pieces]]
    open_form_stack = []   # stack of (code, closing_anchor_id)

    def current_label():
        if not open_form_stack:
            return UNMARKED_FORM_LABEL
        code = open_form_stack[-1][0]
        return code_to_label.get(code, code)

    def emit_text(text):
        if not text:
            return
        label = current_label()
        if segments and segments[-1][0] == label:
            segments[-1][1].append(text)
        else:
            segments.append([label, [text]])

    def walk(elem):
        tag = elem.tag.split("}")[-1]
        if tag == "note":
            return

        if tag == "milestone" and elem.get("unit") == "form":
            code = (elem.get("ana") or "").replace("#", "").strip()
            close_id = (elem.get("spanTo") or "").replace("#", "").strip()
            if code and close_id:
                open_form_stack.append((code, close_id))
        elif tag == "anchor":
            anchor_id = elem.get("{http://www.w3.org/XML/1998/namespace}id")
            # Remove by matching id rather than requiring it to be on top
            # of the stack, to tolerate any imperfectly nested spans.
            open_form_stack[:] = [s for s in open_form_stack if s[1] != anchor_id]

        if elem.text:
            emit_text(elem.text)
        for child in elem:
            walk(child)
            if child.tail:
                emit_text(child.tail)

    walk(body)

    return [
        (label, " ".join("".join(pieces).split()))
        for label, pieces in segments
        if "".join(pieces).strip()
    ]


def parse_tei_file(filepath):
    """
    Parse a single TEI/XML file and return (doc_id, header_metadata, segments):
    - header_metadata: dict with Author, date, periodical_names, language
      (same for every chunk from this file);
    - segments: list of (narrative_form_label, text) tuples, one per
      narrative-form span (plus "unmarked" text in between).
    """
    tree = ET.parse(filepath)
    root = tree.getroot()
    doc_id = os.path.splitext(os.path.basename(filepath))[0]
    header_metadata = parse_tei_header(root)
    segments = extract_form_segments(root)
    return doc_id, header_metadata, segments


print("Parsing helpers defined.")


In [ ]:
if NEEDS_INGESTION:
    # A narrative-form segment usually becomes exactly one chunk. Only
    # unusually long segments (rare -- e.g. a long quoted citation) get
    # further split, purely to keep individual embeddings a reasonable size.
    MAX_CHUNK_CHARS = 2000

    fallback_splitter = RecursiveCharacterTextSplitter(
        chunk_size=MAX_CHUNK_CHARS,
        chunk_overlap=150,
    )

    xml_files = sorted(glob.glob(os.path.join(CORPUS_DIR, "**", "*.xml"), recursive=True))
    print(f"Found {len(xml_files)} TEI/XML file(s) under {CORPUS_DIR!r}")

    all_documents = []

    for filepath in xml_files:
        try:
            doc_id, header_metadata, segments = parse_tei_file(filepath)
        except ET.ParseError as e:
            print(f"Skipping {filepath}: could not parse XML ({e})")
            continue

        if not segments:
            print(f"Skipping {filepath}: no body text found")
            continue

        chunk_index = 0
        for form_label, segment_text in segments:
            pieces = (
                fallback_splitter.split_text(segment_text)
                if len(segment_text) > MAX_CHUNK_CHARS
                else [segment_text]
            )
            for piece in pieces:
                chunk_metadata = dict(header_metadata)
                chunk_metadata["narrative_forms"] = form_label
                chunk_metadata["chunk_id"] = f"{doc_id}_chunk_{chunk_index}"
                chunk_index += 1

                all_documents.append(Document(page_content=piece, metadata=chunk_metadata))

    print(f"Built {len(all_documents)} chunks from {len(xml_files)} source file(s)")


In [ ]:
if NEEDS_INGESTION:
    import time

    MAX_RETRIES = 3
    RETRY_DELAY_SECONDS = 5
    SAFETY_MARGIN = 200  # stay comfortably under Chroma's own max batch size

    try:
        chroma_max_batch = vector_store._client.get_max_batch_size()
    except Exception:
        chroma_max_batch = 500  # conservative fallback if the client doesn't expose this

    BATCH_SIZE = max(1, chroma_max_batch - SAFETY_MARGIN)
    print(f"Using batch size {BATCH_SIZE} (Chroma's max batch size is {chroma_max_batch})")

    if all_documents:
        # Chroma metadata values must be str/int/float/bool; this drops any
        # remaining non-scalar or missing values (e.g. a chunk with no
        # preceding_document) rather than erroring on add_documents.
        all_documents = filter_complex_metadata(all_documents)

        added = 0
        failed_batches = []
        total_batches = (len(all_documents) + BATCH_SIZE - 1) // BATCH_SIZE

        for start in range(0, len(all_documents), BATCH_SIZE):
            batch = all_documents[start:start + BATCH_SIZE]
            batch_num = start // BATCH_SIZE + 1

            for attempt in range(1, MAX_RETRIES + 1):
                try:
                    vector_store.add_documents(batch, ids=[d.metadata["chunk_id"] for d in batch])
                    added += len(batch)
                    print(f"Batch {batch_num}/{total_batches}: added {len(batch)} chunks "
                          f"({added}/{len(all_documents)} total)")
                    break
                except Exception as e:
                    print(f"Batch {batch_num}/{total_batches}, attempt {attempt}/{MAX_RETRIES} "
                          f"failed: {type(e).__name__}: {e}")
                    if attempt < MAX_RETRIES:
                        time.sleep(RETRY_DELAY_SECONDS)
                    else:
                        print(f"Batch {batch_num} permanently failed after "
                              f"{MAX_RETRIES} attempts -- skipping for now.")
                        failed_batches.append(batch_num)

        print()
        print(f"Added {added}/{len(all_documents)} chunks to the Chroma vector store.")
        if failed_batches:
            print(f"Batches that failed and were skipped: {failed_batches}")
            print("Re-run this cell (Chroma will just re-add the same chunk_ids) "
                  "once the underlying issue is resolved.")
    else:
        print("No documents to add -- check CORPUS_DIR and that it contains .xml files.")


In [ ]:
print("Documents in Chroma:", vector_store._collection.count())

In [ ]:
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent


# 1. Define the Chroma retrieval tool.
@tool
def retrieve_context(
    query: str,
    author: str | None = None,
    periodical_name: str | None = None,
    language: str | None = None,
    date: str | None = None,
    narrative_form: str | None = None,
    k: int = 10,
) -> str:
    """
    Retrieve relevant passages from the multilingual corpus of TEI encoded historical
    periodical texts. 

    The query may be in English, German, French, Spanish or another language;
    the corpus itself is multilingual. All filters below are optional and
    should only be applied when the user's question clearly asks for them.

    - author: filter by the author name as given in the source.
      
    - periodical_name: filter by the name of the periodical/journal.
      
    - language: filter by language of the source text (e.g. "French", "German").
        Take into account that there might be cases where the language can appear as a compound of languages(e.g. "Spanish; castilian").
    
    - date: (e.g. "1716"). Note dates may be a single year or a year range
      (e.g. "1711-1712") and are matched as exact strings, not numeric ranges.
      
    - narrative_form: filter by narrative form (e.g. "Dialogue", "Letter/Letter
      to the editor", "Example", "Satire"). Each chunk covers exactly one
      narrative-form span (or "unmarked" for running text outside any
      tagged span).

    - for the k variable don't override the value in retrieved_docs
    """

    # Build optional Chroma metadata filters.
    conditions = []

    if author:
        conditions.append({"Author": {"$eq": author.strip()}})

    if periodical_name:
        conditions.append({"periodical_names": {"$eq": periodical_name.strip()}})

    if language:
        conditions.append({"language": {"$eq": language.strip()}})

    if date:
        conditions.append({"date": {"$eq": date.strip()}})

    if narrative_form:
        conditions.append({"narrative_forms": {"$eq": narrative_form.strip()}})

    # Prepare the final metadata filter.
    if not conditions:
        metadata_filter = None
    elif len(conditions) == 1:
        metadata_filter = conditions[0]
    else:
        metadata_filter = {"$and": conditions}

    # Retrieve documents using semantic similarity and metadata filters.
    retrieved_docs = vector_store.similarity_search(
        query=query,
        k=max(1, min(int(k), 10)),
        filter=metadata_filter,
    )

    if not retrieved_docs:
        return "No relevant passages were found."

    # Convert the retrieved Document objects into text for the agent.
    serialized_documents = []

    for number, doc in enumerate(retrieved_docs, start=1):
        metadata = doc.metadata

        serialized_documents.append(
            f"""
SOURCE {number}

METADATA
Chunk ID: {metadata.get("chunk_id", "unknown")}
Periodical: {metadata.get("periodical_names", "unknown")}
Author: {metadata.get("Author", "unknown")}
Date: {metadata.get("date", "unknown")}
Language: {metadata.get("language", "unknown")}
Narrative form: {metadata.get("narrative_forms", "unknown")}

RETRIEVED TEXT
{doc.page_content or "[No retrieved text]"}
""".strip()
        )

    separator = "\n\n" + "=" * 80 + "\n\n"
    return separator.join(serialized_documents)


# 2. Define the research assistant's behaviour.
system_prompt = """
You are a research assistant working with a multilingual corpus of
historical letters and periodical texts (18th-century moral weeklies),
encoded in TEI/XML.

Always use the retrieve_context tool before answering questions about the
corpus. Base your answer only on the passages returned by the tool.

The corpus spans multiple languages (e.g. French, German, English, Spanish).
The user's question may be in a different language than the source texts;
when useful, formulate a multilingual retrieval query and include
historical spelling or terminology variants.

Extract metadata filters from the user's question:

- use author when the user names a specific author;
- use periodical_name when the user names a specific periodical/journal;
- use language when the user asks about a specific language of source text;
- use date when the user specifies a year or date found in the sources;
- use narrative_form when the user asks about a specific narrative form
  (e.g. dialogue, satire, letter, example, allegory);
- do not invent filters that the user did not request.

In your answer:

- summarize the evidence;
- distinguish evidence from interpretation;
- cite chunk IDs, dates, authors, and periodical names;
- identify differences or contradictions between sources;
- note when sources are in different languages;
- state clearly when the retrieved evidence is insufficient.
"""


# 3. Register the tool and create the agent.
tools = [retrieve_context]

agent = create_react_agent(
    model=model,
    tools=tools,
    prompt=system_prompt,
)

print("Agent created successfully!")


## Search function and example queries

In [ ]:
query = (
    "What it the newest dialogue narrative form in italian and in french"
)

for step in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

In [ ]:
query = (
    "What content does the earliest form of satire inside the corpus have?"
)

for step in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

In [ ]:
query = (
    "Über welche Themen werden in den Satire Abschnitten der Texte diskutiert?"
)

for step in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

In [ ]:
query = (
    "Kommt in diesem Korpus etwas zu Roger De Coverley vor?"
)

for step in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

In [ ]:
query = (
    "Are there some forms of satirical character sketches of aristocrates inside the corpus?"
)

for step in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()